In [19]:
import pandas as pd
import numpy as np
from itertools import product

In [20]:
df = pd.read_csv('Crime_Data_from_2020_to_Present.csv')

In [21]:


lat_bins = np.arange(df['LAT'].min(), df['LAT'].max(), 0.005)  # ~500m steps
lon_bins = np.arange(df['LON'].min(), df['LON'].max(), 0.005)

df['lat_bin'] = np.digitize(df['LAT'], lat_bins)
df['lon_bin'] = np.digitize(df['LON'], lon_bins)
df['grid_id'] = df['lat_bin'].astype(str) + "_" + df['lon_bin'].astype(str)

In [22]:
# Example: daily crime count per grid
df['date'] = pd.to_datetime(df['DATE OCC'])
grid_daily = df.groupby(['grid_id', 'date']).size().reset_index(name='crime_count')

In [23]:
# --- THE FIX: Force the original DATE OCC column to be a datetime object first ---
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'])

# 1. Find all unique grids and the full date range of your dataset
unique_grids = df['grid_id'].unique()
date_range = pd.date_range(start=df['DATE OCC'].min(), end=df['DATE OCC'].max())

# 2. Create the "Blank Canvas" containing every grid on every day
master_grid = pd.DataFrame(list(product(unique_grids, date_range)), columns=['grid_id', 'DATE OCC'])

# 3. Group your actual crimes to get daily counts per grid 
grid_daily = df.groupby(['grid_id', 'DATE OCC']).size().reset_index(name='crime_count')

# 4. Merge the real crimes onto the blank canvas and fill the missing days with 0
# (This merge will now work perfectly because both are datetime64!)
master_data = pd.merge(master_grid, grid_daily, on=['grid_id', 'DATE OCC'], how='left')
master_data['crime_count'] = master_data['crime_count'].fillna(0)

# 5. Define your target: 1 if a crime happened, 0 if it didn't
master_data['crime_happened'] = (master_data['crime_count'] >= 1).astype(int)

In [24]:
# 1. Sort the data chronologically just to be safe
master_data = master_data.sort_values(['grid_id', 'DATE OCC'])

# 2. Add the Memories
master_data['crimes_last_7_days'] = master_data.groupby('grid_id')['crime_count'].transform(lambda x: x.rolling(7, closed='left').sum().fillna(0))
master_data['crimes_last_30_days'] = master_data.groupby('grid_id')['crime_count'].transform(lambda x: x.rolling(30, closed='left').sum().fillna(0))
master_data['crimes_last_90_days'] = master_data.groupby('grid_id')['crime_count'].transform(lambda x: x.rolling(90, closed='left').sum().fillna(0))

# 3. Add the Calendar Features (THE FIX)
master_data['DayOfWeek'] = master_data['DATE OCC'].dt.dayofweek
master_data['Month'] = master_data['DATE OCC'].dt.month
master_data['is_weekend'] = master_data['DayOfWeek'].isin([5, 6]).astype(int)

# 4. Add the Map Coordinates (THE FIX)
grid_coords = df[['grid_id', 'lat_bin', 'lon_bin']].drop_duplicates()
master_data = pd.merge(master_data, grid_coords, on='grid_id', how='left')

In [25]:
master_data.head()

,grid_id,DATE OCC,crime_count,crime_happened,crimes_last_7_days,crimes_last_30_days,crimes_last_90_days,DayOfWeek,Month,is_weekend,lat_bin,lon_bin
0,1_23734,2020-01-01,0.0,0,0.0,0.0,0.0,2,1,0,1,23734
1,1_23734,2020-01-02,0.0,0,0.0,0.0,0.0,3,1,0,1,23734
2,1_23734,2020-01-03,0.0,0,0.0,0.0,0.0,4,1,0,1,23734
3,1_23734,2020-01-04,0.0,0,0.0,0.0,0.0,5,1,1,1,23734
4,1_23734,2020-01-05,0.0,0,0.0,0.0,0.0,6,1,1,1,23734


In [26]:
# 5. Split the Timeline (Last 20% for testing)
split_index = int(len(date_range) * 0.8)
cutoff_date = date_range[split_index]

train_data = master_data[master_data['DATE OCC'] < cutoff_date]
test_data = master_data[master_data['DATE OCC'] >= cutoff_date]

# 6. Define the exact columns the model is allowed to look at
features = [
    'DayOfWeek', 'Month', 'is_weekend', 
    'crimes_last_7_days', 'crimes_last_30_days', 'crimes_last_90_days', 
    'lat_bin', 'lon_bin'
]

X_train = train_data[features]
y_train = train_data['crime_happened']

X_test = test_data[features]
y_test = test_data['crime_happened']

print("Success! Data is prepped and split.")

Success! Data is prepped and split.


In [27]:
from xgboost import XGBClassifier

# Initialize the model
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# Train the model on the historical data
model.fit(X_train, y_train)

# Output the PROBABILITIES, not just 1s and 0s
# [:, 1] grabs the probability of Class 1 (a crime happening)
probabilities = model.predict_proba(X_test)[:, 1]

# Attach it back to your test dataframe so you can see the results!
test_data = test_data.copy()
test_data['predicted_probability'] = probabilities

# Look at the predictions for a specific grid over time
print(test_data[['grid_id', 'DATE OCC', 'crime_happened', 'predicted_probability']].head(10))

      grid_id   DATE OCC  crime_happened  predicted_probability
1530  1_23734 2024-03-10               0               0.006776
1531  1_23734 2024-03-11               0               0.009331
1532  1_23734 2024-03-12               0               0.008343
1533  1_23734 2024-03-13               0               0.009259
1534  1_23734 2024-03-14               0               0.008396
1535  1_23734 2024-03-15               0               0.008417
1536  1_23734 2024-03-16               0               0.008243
1537  1_23734 2024-03-17               0               0.006776
1538  1_23734 2024-03-18               0               0.009331
1539  1_23734 2024-03-19               0               0.008343


In [33]:
# Sort the test data to show the highest predicted probabilities first
top_risks = test_data.sort_values(by='predicted_probability', ascending=False)

# Display the top 15 most dangerous predictions
print(top_risks[['grid_id', 'DATE OCC', 'crime_happened', 'crimes_last_90_days' ,'predicted_probability']].head(15))

         grid_id   DATE OCC  crime_happened  crimes_last_90_days  \
2817633  6812_87 2024-08-24               1                684.0   
2817612  6812_87 2024-08-03               1                465.0   
2817626  6812_87 2024-08-17               1                611.0   
2817619  6812_87 2024-08-10               1                532.0   
2817611  6812_87 2024-08-02               1                463.0   
2817625  6812_87 2024-08-16               1                600.0   
2817632  6812_87 2024-08-23               0                690.0   
2817618  6812_87 2024-08-09               1                531.0   
2817591  6812_87 2024-07-13               1                273.0   
2817598  6812_87 2024-07-20               1                335.0   
2817605  6812_87 2024-07-27               1                408.0   
2817617  6812_87 2024-08-08               1                531.0   
2817624  6812_87 2024-08-15               1                591.0   
2817610  6812_87 2024-08-01               1     

In [34]:
import joblib
# This saves your trained XGBoost model as a file in your folder
joblib.dump(model, 'crime_predictor_model.joblib')

['crime_predictor_model.joblib']